In [ ]:
import json
import sys
from typing import List, Optional

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from simulate_markov_utils import plot_markov_df
from sklearn.metrics import auc, roc_curve

from plotting_notebooks.plotting_utils import (
    load_in_combined_results,
    load_monitoring_data,
    plot_markov_analysis,
    plot_roc_curves,
    plot_suspiciousness_distributions,
)

sys.path.append('..')
from utils.analysis_utils import analyze_all_tasks, extract_verdict
from utils.path_utils import resolve_path

# Constants
MAIN_AGENT_MODELS = ['gpt-4o', 'claude-3-5-sonnet-20241022']
MONITOR_AGENT_MODELS = ['gpt-4o', 'claude-3-5-sonnet-20241022', 'o3-mini', 'o1', 'claude-3-5-haiku-20241022']
DEFAULT_TRANSCRIPT_TYPES = ["transcripts", "transcripts_no_hint", "transcripts_no_secondary_task"]
DOWNSAMPLE_FACTORS = [1, 2, 3, 4, 5, 10]

In [ ]:
def prepare_review_allegations_data(base_dir: str, include_failed_attempts: bool, include_failed_main_task: bool) -> tuple[pd.DataFrame, pd.DataFrame]:
    """Load and prepare the initial datasets."""
    # Load monitoring data
    review_allegations_df = load_monitoring_data(base_dir=base_dir)
    for i in range(1, 4):
        review_allegations_df[f'response_{i}'] = review_allegations_df.data.apply(
            lambda x: extract_verdict(x[f'response_{i}'], tag_word='verdict')
        )
    review_allegations_df['average_suspiciousness'] = (review_allegations_df['response_1'] + review_allegations_df['response_2'] + review_allegations_df['response_3']) / 3
    review_allegations_df['max_suspiciousness'] = np.max([review_allegations_df['response_1'], review_allegations_df['response_2'], review_allegations_df['response_3']], axis=0)
    return review_allegations_df

def merge_and_compute_features_for_review_allegations(combined_results: pd.DataFrame, 
                             review_allegations_df: pd.DataFrame) -> pd.DataFrame:
    """Merge the dataframes and compute additional features."""
    merged_df = pd.merge(
        review_allegations_df,
        combined_results,
        on=['task_name', 'transcript_type', 'main_agent', 'monitor_agent', 'file_name'],
        how='left'
    ).dropna()
    
    # Compute features

    merged_df['convo_length'] = merged_df.transcript.apply(lambda x: len(x['conversation']))
    
    return merged_df


In [ ]:

include_failed_attempts = False
include_failed_main_task = False
combined_results = load_in_combined_results(include_failed_attempts, include_failed_main_task)
review_allegations_df = prepare_review_allegations_data(base_dir='monitor_responses/review_allegations_sp1', include_failed_attempts=include_failed_attempts, include_failed_main_task=include_failed_main_task)
training_data_path = resolve_path('finetuning/training_data_dfs/combined_3.csv')
training_data = pd.read_csv(training_data_path)
review_allegations_df = review_allegations_df[~review_allegations_df.task_name.isin(training_data.task_name)]
review_allegations_df['side_task_evaluation'] = review_allegations_df.transcript_type != 'transcripts_no_secondary_task'

In [ ]:
review_allegations_df['average_suspiciousness'] = (review_allegations_df['response_1'] + review_allegations_df['response_2'] + review_allegations_df['response_3']) / 3
merged_df = merge_and_compute_features_for_review_allegations(combined_results, review_allegations_df)
plot_roc_curves(review_allegations_df[review_allegations_df.task_name == 'election_tampering'], combined_results, give_hint=None, include_failed_attempts=include_failed_attempts, include_failed_main_task=include_failed_main_task, monitors = review_allegations_df.monitor_agent.unique(), metrics = ['response_1', 'response_2', 'response_3', 'average_suspiciousness', 'max_suspiciousness'])
plot_suspiciousness_distributions(review_allegations_df[review_allegations_df.task_name == 'election_tampering'], monitors = review_allegations_df.monitor_agent.unique(), metrics = ['average_suspiciousness', 'max_suspiciousness'])

In [ ]:
merged_df['1_minus_2'] = merged_df['response_1'] - merged_df['response_2']
merged_df['1_minus_3'] = merged_df['response_1'] - merged_df['response_3']
merged_df['2_minus_3'] = merged_df['response_2'] - merged_df['response_3']
plot_suspiciousness_distributions(merged_df, monitors = ['gpt-4o'], metrics = ['1_minus_2', '1_minus_3', '2_minus_3'])

In [ ]:
review_allegations_df.task_name.unique()